# Python Debugging in 2026:
## From Tracebacks to Live Process Attach (new in 3.14)


## Why debugging feels hard

- Bugs are rarely where they appear
- Reproduction is half the battle
- State changes over time
- Concurrency makes everything worse

Most wasted time comes from chasing symptoms instead of causes.


## The 3 questions of debugging

1. What happened?
2. Why did it happen?
3. How do I reproduce it?


## A debugging maturity ladder

`print()` → traceback → `breakpoint()` → `pdb` → logging → monitoring → live attach


## Tracebacks are your first debugger (after print)

- Read the exception type
- Read the bottom frame first
- Then move upward into your code
- Separate symptom from root cause


In [ ]:
def get_cart(user):
    if user == 'guest':
        return None
    return [{'price': 10}, {'price': 20}]

def checkout(user):
    cart = get_cart(user)
    return sum(item['price'] for item in cart)

# Run this to produce a traceback for the demo:
checkout('guest')


## Notebook-friendly version of python `pdb` debugger

live notebook flow:

1. Cause an exception
2. Run `%debug`
3. Inspect state in post-mortem mode

Note:
From a notebook %debug will let you inspect the state of the program as it was immediately after a crash

From the command line, if you run your program with `python -m pdb myscript.py` it will run
until a crash and then put you in the debugger to inspect the state of the program

In [ ]:
# After running the failing cell above, run this in Jupyter:
%debug


## `breakpoint()` and `pdb.set_trace()`

Core commands:

- `n` next
- `s` step
- `c` continue
- `p expr` print expression
- `w` stack trace
- `q` quit


In [ ]:
import pdb

def buggy_total(items):
    total = 0
    for item in items:
        pdb.set_trace()
        total += item['price']
    return total

# Run this and use: n, p item, p total, c
buggy_total([{'price': 10}, {'price': 5}])


## How to use a debugger well

**Bad**
- step every line
- inspect random variables

**Good**
- start with a hypothesis
- stop near a suspicious boundary
- compare expected vs actual state


## Conditional breakpoints

Break only when:

- `user_id == 42`
- `x is None`
- `len(queue) > 1000`


## Post-mortem debugging

- crash first
- inspect after failure
- useful for long-running jobs
- useful for tests


## Debugging async code

- work pauses and resumes
- state changes between `await`s
- exceptions may surface far away
- timing becomes part of the bug


In [ ]:
import asyncio

state = {'count': 0}

async def worker():
    tmp = state['count']
    await asyncio.sleep(0)
    state['count'] = tmp + 1

async def main():
    await asyncio.gather(*(worker() for _ in range(100)))
    print(state)

asyncio.run(main())


## Logging vs debugging vs profiling

- Debugger: best for local exact state
- Logging: best across time or machines
- Profiler: best for slow code
- Monitoring: best for runtime visibility


## Crash and hang debugging

- stack dumps
- `faulthandler`
- inspect deadlocks and hangs
- useful when no Python exception appears


In [ ]:
import faulthandler
faulthandler.enable()
print('faulthandler enabled')


## Modern debugging internals

- `bdb`
- `pdb`
- `sys.monitoring`
- lower-overhead instrumentation


## The big new thing: live process attach

**Python 3.14**

- safe external debugger interface
- attach to a running process
- zero overhead until used
- exposed via `sys.remote_exec()`
- New tools can attach to a running Python process and run arbitrary code INSIDE that program's context
- You can attach, and inspect variables in real-time. The tools have yet to be updated to use this.


## Toy example (but wait for better tools)

In [ ]:
import time

counter = 0
while True:
    counter += 1
    time.sleep(1)

In [ ]:
import sys
from pathlib import Path

# run this code IN THE OTHER process:
Path("inspect_target.py").write_text("""
import builtins
print("Attached successfully")
print("counter =", globals().get("counter"))
""")

sys.remote_exec(1234, "inspect_target.py")

## A practical debugging workflow

1. Read traceback
2. Form hypothesis
3. Reproduce minimally
4. Use `breakpoint()` / `pdb`
5. Add selective logging
6. Use post-mortem or runtime tools for harder cases
